# Exercise 1: Build the CSVReader Class (OOP Foundation)

## Objective

- Reinforce OOP fundamentals: classes, constructors, methods, and encapsulation.
- Learn to design reusable data classes for Data Science projects.

## Concept

A `CSVReader` class should encapsulate all functionality related to reading CSV files.

## Instructions

1. Create a class named `CSVReader`.
2. Define an `__init__()` method that takes `file_path`.
3. Add a `read()` method that loads the CSV file using pandas.
4. Add a `preview(n)` method that prints the top `n` rows.

## Step 1 — Create the `CSVReader` Class

The class contains:

- `__init__(file_path)` → stores the CSV file path. A constructor is a special method that runs automatically when you create a new object.
- `read()` → reads the CSV file using pandas.
- `preview(n)` → prints the first `n` rows.

`self.data` stores the loaded DataFrame so that other methods can reuse it.

In [2]:
class CSVReader:
    """A simple reusable class for reading and previewing CSV files."""

    def __init__(self, file_path):
        """Store the path of the CSV file."""
        self.file_path = file_path
        self.data = None

    def read(self):
        """Read the CSV file and store it in self.data."""
        self.data = pd.read_csv(self.file_path)
        return self.data

    def preview(self, n=5):
        """Print the first n rows of the loaded CSV file."""
        if self.data is None:
            self.read()

        print(self.data.head(n))

## Step 2 — Create an Object

The CSV file is in the same folder as this notebook, so we pass its filename to `CSVReader`.

In [3]:
reader = CSVReader("sample_data (2).csv")

## Step 3 — Read the CSV File

Calling `read()` loads the CSV file and returns a pandas DataFrame.

In [4]:
import pandas as pd
df = reader.read()
df

,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,NaN,172.0,NaN,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,NaN
3,4,David,NaN,NaN,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,NaN
6,7,Grace,NaN,162.0,60.0,Chicago,92.0


## Step 4 — Preview the First 5 Rows

The `preview()` method displays the top rows of the dataset.  
By default, `n=5`.

In [5]:
reader.preview(5)

   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN
3   4    David   NaN        NaN       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0


## Step 5 — Preview a Different Number of Rows

Because `preview(n)` accepts an argument, we can choose how many rows to display.

In [6]:
reader.preview(3)

   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN


## Result

The `CSVReader` class successfully:

1. Stores the CSV file path.
2. Reads the file with pandas.
3. Stores the DataFrame inside the object.
4. Previews any requested number of rows.

# Exercise 2: Apply Pattern for Missing Values Handling

## Objective

- Learn the Strategy Pattern: one interface, many interchangeable behaviors.
- Practice inheritance and polymorphism.

## Concept

Different strategies can be used to handle missing data, such as dropping missing values, filling them with the mean, or filling them with the mode.

The Strategy Pattern allows us to switch between these methods without changing the main code.

## Instructions:
1. Create an abstract class MissingValueStrategy with a method handle(df).
2. Create subclasses:
- oDropMissing
- oFillMean
- oFillMode
3. Create a DataCleaner class that applies the chosen strategy.

## Step 1 — Create an abstract class MissingValueStrategy with a method handle(df).

In [7]:
from abc import ABC, abstractmethod


class MissingValueStrategy(ABC):

    @abstractmethod
    def handle(self, df):
        pass

## Step 2 — Create subclasses:
DropMissing

In [8]:
class DropMissing(MissingValueStrategy):
    def handle(self, df):
        return df.dropna()

FillMean

In [9]:
class FillMean(MissingValueStrategy):
    def handle(self, df):
        result = df.copy()
        numeric_cols = result.select_dtypes(include="number").columns
        result[numeric_cols] = result[numeric_cols].fillna(
            result[numeric_cols].mean()
        )
        return result

FillMode

In [10]:
class FillMode(MissingValueStrategy):
    def handle(self, df):
        result = df.copy()

        for column in result.columns:
            if result[column].isna().any():
                mode_values = result[column].mode()

                if not mode_values.empty:
                    result[column] = result[column].fillna(
                        mode_values.iloc[0]
                    )

        return result

## Step 3 — Create a DataCleaner class that applies the chosen strategy.

In [11]:
class DataCleaner:
    def __init__(self, strategy):
        self.strategy = strategy

    def clean(self, df):
        return self.strategy.handle(df)

## Practice: Apply three strategies (DropMissing, FillMean, and FillMode) to handle missing values in the dataset and compare their results.

In [12]:
# 1. Drop Missing Values
cleaner1 = DataCleaner(DropMissing())
result1 = cleaner1.clean(df)

print("\n========== DropMissing ==========")
print(result1.head())


# 2.Fill Missing Values with Mean
cleaner2 = DataCleaner(FillMean())
result2 = cleaner2.clean(df)

print("\n========== FillMean ==========")
print(result2.head())


# 3.Fill Missing Values with Mode
cleaner3 = DataCleaner(FillMode())
result3 = cleaner3.clean(df)

print("\n========== FillMode ==========")
print(result3.head())


========== DropMissing ==========
   id   name   age  height_cm  weight_kg      city  score
0   1  Alice  29.0      165.0       68.0  New York   85.0
4   5    Eva  27.0      160.0       55.0  New York   88.0

========== FillMean ==========
   id     name    age  height_cm  weight_kg         city  score
0   1    Alice  29.00      165.0       68.0     New York   85.0
1   2      Bob  30.25      172.0       70.0  Los Angeles   90.0
2   3  Charlie  35.00      168.0       72.0      Chicago   86.0
3   4    David  30.25      167.0       80.0      Houston   75.0
4   5      Eva  27.00      160.0       55.0     New York   88.0

========== FillMode ==========
   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob  27.0      172.0       55.0  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago   75.0
3   4    David  27.0      160.0       80.0      Houston   75.0
4   5      Eva  27.0      16

# Exercise 3: Add Decorators for Logging and Timing

**Objectives**
- Learn the **Decorator Pattern** to add reusable behaviour (logging, timing).
- Understand how decorators support the **Open/Closed Principle**: classes stay *closed for modification* but *open for extension*.

**Concept:** a decorator is a function that takes a function, wraps it with extra behaviour, and returns the wrapper. The original method body is never edited.

In [13]:
import sys
import time
import logging
import functools
import pandas as pd

# Configure logging once so the decorators print to the notebook
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
    stream=sys.stdout,   # same stream as print() so output stays in order
    force=True,
)
logger = logging.getLogger("pipeline")

## Step 1 — Create the two decorators

- `@log_action` → logs when a method **starts** and **finishes** (and logs errors if it fails).
- `@log_time` → measures and prints the **execution time**.

`functools.wraps` keeps the original function's name and docstring, so `func.__name__` still says `read_csv` instead of `wrapper`.

In [14]:
def log_action(func):
    """Log when a function/method starts and finishes."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        logger.info(f"▶ START  {func.__qualname__}")
        try:
            result = func(*args, **kwargs)
        except Exception as e:
            logger.error(f"✖ FAILED {func.__qualname__}: {e}")
            raise
        logger.info(f"✔ FINISH {func.__qualname__}")
        return result
    return wrapper


def log_time(func):
    """Measure and print how long a function/method takes."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"⏱ {func.__qualname__} took {elapsed * 1000:.3f} ms")
        return result
    return wrapper

### Quick test on a plain function

In [15]:
@log_action
@log_time
def slow_add(a, b):
    time.sleep(0.2)
    return a + b

print("Result:", slow_add(2, 3))
print("Name preserved by functools.wraps:", slow_add.__name__)

10:17:22 | INFO | ▶ START  slow_add
⏱ slow_add took 198.035 ms
10:17:23 | INFO | ✔ FINISH slow_add
Result: 5
Name preserved by functools.wraps: slow_add


## Step 2 — Apply the decorators to `CSVReader` and `DataCleaner`

The class logic is exactly the same as before — the only change is the decorator lines above each method. That is the Open/Closed Principle in action.

In [16]:
class CSVReader:
    """Responsible only for loading CSV data."""

    def __init__(self, file_path):
        self.file_path = file_path

    @log_action
    @log_time
    def read_csv(self):
        df = pd.read_csv(self.file_path)
        print(f"Loaded {df.shape[0]} rows × {df.shape[1]} columns")
        return df


class DataCleaner:
    """Responsible only for cleaning data."""

    def __init__(self, df):
        self.df = df.copy()

    @log_action
    @log_time
    def show_missing(self):
        missing = self.df.isnull().sum()
        print("Missing values per column:")
        print(missing[missing > 0])
        return missing

    @log_action
    @log_time
    def fill_missing_with_mean(self):
        num_cols = self.df.select_dtypes(include="number").columns
        self.df[num_cols] = self.df[num_cols].fillna(self.df[num_cols].mean().round(1))
        return self.df

    @log_action
    @log_time
    def remove_duplicates(self):
        before = len(self.df)
        self.df = self.df.drop_duplicates()
        print(f"Removed {before - len(self.df)} duplicate rows")
        return self.df

## Step 3 — Run the pipeline and watch the logs

In [17]:
reader = CSVReader("D:\Advanced-Programming-TP\sample_data (2).csv")
raw_df = reader.read_csv()
raw_df

10:17:23 | INFO | ▶ START  CSVReader.read_csv
Loaded 7 rows × 7 columns
⏱ CSVReader.read_csv took 3.258 ms
10:17:23 | INFO | ✔ FINISH CSVReader.read_csv


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,NaN,172.0,NaN,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,NaN
3,4,David,NaN,NaN,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,NaN
6,7,Grace,NaN,162.0,60.0,Chicago,92.0


In [18]:
cleaner = DataCleaner(raw_df)
cleaner.show_missing()
cleaner.fill_missing_with_mean()
clean_df = cleaner.remove_duplicates()
clean_df

10:17:23 | INFO | ▶ START  DataCleaner.show_missing
Missing values per column:
age          3
height_cm    1
weight_kg    1
score        2
dtype: int64
⏱ DataCleaner.show_missing took 1.620 ms
10:17:23 | INFO | ✔ FINISH DataCleaner.show_missing
10:17:23 | INFO | ▶ START  DataCleaner.fill_missing_with_mean
⏱ DataCleaner.fill_missing_with_mean took 4.862 ms
10:17:23 | INFO | ✔ FINISH DataCleaner.fill_missing_with_mean
10:17:23 | INFO | ▶ START  DataCleaner.remove_duplicates
Removed 0 duplicate rows
⏱ DataCleaner.remove_duplicates took 6.873 ms
10:17:23 | INFO | ✔ FINISH DataCleaner.remove_duplicates


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,30.2,172.0,70.0,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,86.0
3,4,David,30.2,167.0,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,86.0
6,7,Grace,30.2,162.0,60.0,Chicago,92.0


In [19]:
print("Missing values after cleaning:", clean_df.isnull().sum().sum())

Missing values after cleaning: 0


## Step 4 — Open/Closed Principle without touching the class

We can even add timing/logging to a class **we are not allowed to edit** by wrapping its method from the outside:

In [20]:
class Reporter:
    def summary(self, df):
        return df.describe().round(2)

# Extend behaviour from the outside — Reporter's source is unchanged
Reporter.summary = log_action(log_time(Reporter.summary))

Reporter().summary(clean_df)

10:17:23 | INFO | ▶ START  Reporter.summary


⏱ Reporter.summary took 17.248 ms
10:17:23 | INFO | ✔ FINISH Reporter.summary


,id,age,height_cm,weight_kg,score
count,7.00,7.00,7.00,7.00,7.00
mean,4.00,30.23,167.00,70.00,86.00
std,2.16,2.41,5.29,10.47,5.45
min,1.00,27.00,160.00,55.00,75.00
25%,2.50,29.50,163.50,64.00,85.50
50%,4.00,30.20,167.00,70.00,86.00
75%,5.50,30.20,170.00,76.00,89.00
max,7.00,35.00,175.00,85.00,92.00


## Summary

| Decorator | What it adds | Applied to |
|---|---|---|
| `@log_action` | Start / finish (and error) log messages | `read_csv`, `show_missing`, `fill_missing_with_mean`, `remove_duplicates` |
| `@log_time` | Execution time in ms | same methods |

**Why this follows Open/Closed:** logging and timing were added without modifying the bodies of `CSVReader` or `DataCleaner`. The decorators are reusable on any function, and removing them is as easy as deleting one line.

**Stacking order:** with `@log_action` on top of `@log_time`, `log_action` is the outer wrapper — its START prints first and FINISH prints last, and the timer measures only the method itself.

# Exercise 4: Implement Factory Pattern for Data Transformations

## Objective

- Practice the **Factory Pattern** for scalable creation of transformation objects.
- Apply **abstraction** and **composition**.

## Concept

The Factory Pattern creates transformation objects based on a given input instead of manually creating each object. This makes the code easier to maintain, extend, and reuse.

## Instructions

1. Create an abstract class named `DataTransform` with an `apply(df)` method.
2. Implement the following subclasses:
   - `NormalizeColumns`
   - `RemoveDuplicates`
   - `StandardizeText`
3. Create a `TransformFactory` class that returns transformation objects based on a string input.

In [26]:
import pandas as pd
from abc import ABC, abstractmethod

class DataTransform(ABC):
    @abstractmethod
    # Purpose: Forces subclasses to implement a method. Prevents instantiation of the base class.
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        """Return a transformed copy of the input DataFrame."""
        raise NotImplementedError


class NormalizeColumns(DataTransform):
    """Normalize numeric columns to the range [0, 1]."""

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        result = df.copy()
        for column in result.select_dtypes(include='number').columns:
            minimum = result[column].min()
            maximum = result[column].max()
            if pd.notna(minimum) and pd.notna(maximum) and maximum != minimum:
                result[column] = (result[column] - minimum) / (maximum - minimum)
            elif pd.notna(minimum):
                result[column] = 0.0
        return result


class RemoveDuplicates(DataTransform):
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.drop_duplicates().reset_index(drop=True)


class StandardizeText(DataTransform):
    # text column: applies strip().lower() to string values
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        result = df.copy()
        for column in result.select_dtypes(include=['object', 'string']).columns:
            result[column] = result[column].map(
                lambda value: value.strip().lower() if isinstance(value, str) else value
            )
        return result


class TransformFactory:
    #One registry maps string names to classes, letting you create transformations dynamically without hardcoding imports or if/else chains.
    _TRANSFORMS = {
        'normalize': NormalizeColumns,
        'remove_duplicates': RemoveDuplicates,
        'duplicates': RemoveDuplicates,
        'standardize_text': StandardizeText,
        'text': StandardizeText,
    }

    @classmethod
    # Purpose: Factory method to create instances of DataTransform subclasses based on a string name.
    def create(cls, transform_name: str) -> DataTransform:
        key = transform_name.strip().lower()
        try:
            return cls._TRANSFORMS[key]()
        # Handle the case where the provided transform name is not in the registry.
        except KeyError as error:
            supported = ', '.join(sorted(cls._TRANSFORMS))
            raise ValueError(
            # Provide a helpful error message with the list of supported transformation names.
                f"Unknown transformation '{transform_name}'. Supported names: {supported}"
            ) from error

In [29]:
# Use the FillMean strategy from Exercise 2
cleaned_data = FillMean().handle(df)

# Use the fully cleaned DataFrame
cleaned_input = cleaned_data.copy()

print("Missing values:", cleaned_input.isna().sum().sum())

# Apply each transformation separately
normalized_df = TransformFactory.create('normalize').apply(cleaned_input)
deduplicated_df = TransformFactory.create('remove_duplicates').apply(cleaned_input)
standardized_df = TransformFactory.create('text').apply(cleaned_input)

print('\nNormalized numeric columns:')
display(normalized_df)

print('\nDuplicate rows removed:')
display(deduplicated_df)

print('\nText columns standardized:')
display(standardized_df)

# Compose transformations into a pipeline
pipeline = ['text', 'normalize', 'remove_duplicates']
transformed_df = cleaned_input.copy()

for transform_name in pipeline:
    transformed_df = TransformFactory.create(transform_name).apply(transformed_df)

print('\nComposed pipeline result:')
display(transformed_df)

Missing values: 0

Normalized numeric columns:


,id,name,age,height_cm,weight_kg,city,score
0,0.000000,Alice,0.25000,0.333333,0.433333,New York,0.588235
1,0.166667,Bob,0.40625,0.800000,0.500000,Los Angeles,0.882353
2,0.333333,Charlie,1.00000,0.533333,0.566667,Chicago,0.647059
3,0.500000,David,0.40625,0.466667,0.833333,Houston,0.000000
4,0.666667,Eva,0.00000,0.000000,0.000000,New York,0.764706
5,0.833333,Frank,0.37500,1.000000,1.000000,Los Angeles,0.647059
6,1.000000,Grace,0.40625,0.133333,0.166667,Chicago,1.000000



Duplicate rows removed:


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.00,165.0,68.0,New York,85.0
1,2,Bob,30.25,172.0,70.0,Los Angeles,90.0
2,3,Charlie,35.00,168.0,72.0,Chicago,86.0
3,4,David,30.25,167.0,80.0,Houston,75.0
4,5,Eva,27.00,160.0,55.0,New York,88.0
5,6,Frank,30.00,175.0,85.0,Los Angeles,86.0
6,7,Grace,30.25,162.0,60.0,Chicago,92.0



Text columns standardized:


,id,name,age,height_cm,weight_kg,city,score
0,1,alice,29.00,165.0,68.0,new york,85.0
1,2,bob,30.25,172.0,70.0,los angeles,90.0
2,3,charlie,35.00,168.0,72.0,chicago,86.0
3,4,david,30.25,167.0,80.0,houston,75.0
4,5,eva,27.00,160.0,55.0,new york,88.0
5,6,frank,30.00,175.0,85.0,los angeles,86.0
6,7,grace,30.25,162.0,60.0,chicago,92.0



Composed pipeline result:


,id,name,age,height_cm,weight_kg,city,score
0,0.000000,alice,0.25000,0.333333,0.433333,new york,0.588235
1,0.166667,bob,0.40625,0.800000,0.500000,los angeles,0.882353
2,0.333333,charlie,1.00000,0.533333,0.566667,chicago,0.647059
3,0.500000,david,0.40625,0.466667,0.833333,houston,0.000000
4,0.666667,eva,0.00000,0.000000,0.000000,new york,0.764706
5,0.833333,frank,0.37500,1.000000,1.000000,los angeles,0.647059
6,1.000000,grace,0.40625,0.133333,0.166667,chicago,1.000000


## Exercise 5 · A complete CSV cleaning pipeline

Build a reusable workflow with four design patterns:

| Pattern | Responsibility |
| :--- | :--- |
| **Template Method** | `DataPipeline.run()` fixes the order: load → clean → transform → save. |
| **Strategy** | `DataCleaner` delegates missing-value handling to an interchangeable strategy. |
| **Factory** | `TransformFactory` creates transformations from configuration names. |
| **Decorator** | `@log_action` and `@log_time` provide optional execution diagnostics. |

Run all cells in order. The example writes `dirty_sample.csv` and `cleaned_sample.csv`
in the notebook's working directory. Verbose diagnostics are disabled by default.

### Step 1. Imports and execution decorators

Set `ENABLE_LOGGING = True` to inspect calls, failures, and elapsed time.
Both decorators preserve the wrapped function's metadata; timing is reported even
when a call fails.

In [30]:
from abc import ABC, abstractmethod
from functools import wraps
from pathlib import Path
from time import perf_counter

import pandas as pd
from IPython.display import display

ENABLE_LOGGING = False


def log_action(func):
    """Optionally report the start, success, or failure of a call."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        if ENABLE_LOGGING:
            print(f"[LOG] {func.__qualname__} started")
        try:
            result = func(*args, **kwargs)
        except Exception as exc:
            if ENABLE_LOGGING:
                print(f"[LOG] {func.__qualname__} failed: {exc}")
            raise
        if ENABLE_LOGGING:
            print(f"[LOG] {func.__qualname__} finished")
        return result
    return wrapper


def log_time(func):
    """Optionally report elapsed time, including unsuccessful calls."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        if not ENABLE_LOGGING:
            return func(*args, **kwargs)
        start = perf_counter()
        try:
            return func(*args, **kwargs)
        finally:
            print(f"[TIME] {func.__qualname__}: {perf_counter() - start:.4f}s")
    return wrapper

### Step 2. CSV reader

Encapsulate CSV ingestion and provide a notebook-friendly preview.

In [31]:
class CSVReader:
    def __init__(self, file_path: str | Path):
        self.file_path = Path(file_path)
        self.df: pd.DataFrame | None = None

    @log_action
    @log_time
    def read(self) -> pd.DataFrame:
        self.df = pd.read_csv(self.file_path)
        return self.df

    def preview(self, n: int = 5) -> None:
        if self.df is None:
            raise RuntimeError("Call read() before preview().")
        display(self.df.head(n))

### Step 3. Missing-value strategies · Strategy Pattern

- **`DropMissing`** removes rows containing any missing cell.
- **`FillMean`** fills numeric columns with their means and preserves missing text.
- **`FillMode`** fills each column with its first mode; ties follow pandas' mode ordering.

All strategies return a new DataFrame. A column containing only missing values has
no mean or mode, so it remains missing. Empty or whitespace-only strings are not
treated as missing by these strategies.

In [32]:
class MissingValueStrategy(ABC):
    @abstractmethod
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        """Return cleaned data without changing the input DataFrame."""
        ...


class DropMissing(MissingValueStrategy):
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.dropna().reset_index(drop=True)


class FillMean(MissingValueStrategy):
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.fillna(df.mean(numeric_only=True))


class FillMode(MissingValueStrategy):
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        # Only calculate modes for columns that need imputation.
        replacements = {}
        for name in df.columns[df.isna().any()]:
            modes = df[name].mode()
            if not modes.empty:
                replacements[name] = modes.iloc[0]
        return df.fillna(replacements)

### Step 4. Cleaning context

`DataCleaner` delegates to the selected strategy. With no strategy, it leaves the
data unchanged and avoids allocating an unnecessary copy.

In [33]:
class DataCleaner:
    def __init__(self, strategy: MissingValueStrategy | None = None):
        self.strategy = strategy

    @log_action
    @log_time
    def clean(self, df: pd.DataFrame) -> pd.DataFrame:
        return df if self.strategy is None else self.strategy.handle(df)

### Step 5. Reusable transformations

- **`NormalizeColumns`** strips headers, lowercases them, and collapses whitespace
  to underscores. It rejects headers that would become ambiguous duplicates.
- **`StandardizeText`** strips and lowercases text while preserving missing values
  and non-string values in mixed object columns.
- **`RemoveDuplicates`** removes identical rows and resets the index.

Order matters: standardize text before removing duplicates so cosmetic differences
do not hide duplicate records. Each transformation preserves its input.

In [34]:
class DataTransform(ABC):
    @abstractmethod
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        """Return transformed data without changing the input DataFrame."""
        ...


class NormalizeColumns(DataTransform):
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        columns = (pd.Index(df.columns).astype(str).str.strip().str.lower()
                   .str.replace(r"\s+", "_", regex=True))
        if columns.has_duplicates:
            duplicates = columns[columns.duplicated()].unique().tolist()
            raise ValueError(f"Column normalization creates duplicate names: {duplicates}")
        return df.set_axis(columns, axis="columns")


class RemoveDuplicates(DataTransform):
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.drop_duplicates().reset_index(drop=True)


class StandardizeText(DataTransform):
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        result = df.copy()
        for name in result.select_dtypes(include=["object", "string"]).columns:
            column = result[name]
            if pd.api.types.infer_dtype(column, skipna=True) in {"string", "empty"}:
                result[name] = column.str.strip().str.lower()
            else:
                # The .str accessor alone would replace non-string values with NaN.
                result[name] = column.map(
                    lambda value: value.strip().lower() if isinstance(value, str) else value
                )
        return result

### Step 6. Transformation factory · Factory Pattern

Create the requested transformation from its name. Unknown names raise a clear error.

In [35]:
class TransformFactory:
    _transforms = {
        "normalize_columns": NormalizeColumns,
        "standardize_text": StandardizeText,
        "remove_duplicates": RemoveDuplicates,
    }

    @classmethod
    def get_transform(cls, name: str) -> DataTransform:
        key = name.strip().lower()
        if key not in cls._transforms:
            available = ", ".join(cls._transforms)
            raise ValueError(f"Unknown transform {name!r}. Available: {available}")
        return cls._transforms[key]()

### Step 7. Pipeline template · Template Method Pattern

The base class defines the workflow once. Concrete subclasses implement the four
steps. `run()` returns the pipeline so callers can inspect its result and summary.

In [36]:
class DataPipeline(ABC):
    @log_action
    @log_time
    def run(self):
        """Execute the fixed load → clean → transform → save sequence."""
        self.load()
        self.clean()
        self.transform()
        self.save()
        return self

    @abstractmethod
    def load(self) -> None:
        ...

    @abstractmethod
    def clean(self) -> None:
        ...

    @abstractmethod
    def transform(self) -> None:
        ...

    @abstractmethod
    def save(self) -> None:
        ...

### Step 8. Concrete CSV pipeline

Implement the four template steps using the reader, cleaner, and transformation factory.

In [37]:
class CSVDataPipeline(DataPipeline):
    def __init__(self, input_path, output_path, missing_strategy=None, transforms=None):
        self.input_path = input_path
        self.output_path = output_path
        self.cleaner = DataCleaner(missing_strategy)
        self.transforms = tuple(transforms) if transforms is not None else ()
        self.df = None

    def load(self):
        self.df = CSVReader(self.input_path).read()

    def clean(self):
        self.df = self.cleaner.clean(self.df)

    def transform(self):
        for name in self.transforms:
            transformer = TransformFactory.get_transform(name)
            self.df = transformer.apply(self.df)

    def save(self):
        self.df.to_csv(self.output_path, index=False)

### Step 9. Create the sample dataset

The five input rows contain inconsistent headers and text casing, one duplicate
Alice record, and three missing cells: age, salary, and department.

In [38]:
INPUT_PATH = Path("dirty_sample.csv")
OUTPUT_PATH = Path("cleaned_sample.csv")

raw_df = pd.DataFrame({
    " Full Name ": [" Alice ", "BOB", "Charlie ", " Alice ", "David"],
    " Age ": [25.0, None, 30.0, 25.0, 40.0],
    " Salary ": [50000.0, 60000.0, None, 50000.0, 75000.0],
    " Department ": [" HR ", "engineering", "FINANCE", " HR ", None],
})
raw_df.to_csv(INPUT_PATH, index=False)

### Step 10. Run the pipeline

Run the four steps and compare the original and cleaned data using standard pandas tables.

`FillMean` fills age with **30** and salary with **58,750** before deduplication.
David's department remains missing because it is not numeric.

In [39]:
pipeline = CSVDataPipeline(
    input_path=INPUT_PATH,
    output_path=OUTPUT_PATH,
    missing_strategy=FillMean(),
    transforms=["normalize_columns", "standardize_text", "remove_duplicates"],
).run()

cleaned_df = pipeline.df
print("Before cleaning:")
display(raw_df)
print("After cleaning:")
display(cleaned_df)

Before cleaning:


,Full Name,Age,Salary,Department
0,Alice,25.0,50000.0,HR
1,BOB,NaN,60000.0,engineering
2,Charlie,30.0,NaN,FINANCE
3,Alice,25.0,50000.0,HR
4,David,40.0,75000.0,None


After cleaning:


,full_name,age,salary,department
0,alice,25.0,50000.0,hr
1,bob,30.0,60000.0,engineering
2,charlie,30.0,58750.0,finance
3,david,40.0,75000.0,NaN
